# V10 — Scientific Synthesis

## Purpose

This notebook integrates the computational and empirical results developed throughout the tennis-serve admissibility study.

The synthesis combines:

1. Numerical verification of the trajectory solver.
2. Aerodynamic drag and Magnus-force modeling.
3. Court and service-box constraints.
4. Speed-dependent admissibility envelopes.
5. Spin-dependent admissibility envelopes.
6. Robustness and uncertainty analysis.
7. Empirical analysis of 2024 US Open serve-speed data.

The objective is not to introduce a new physical model, but to evaluate whether the individual analyses form a coherent computational framework and to summarize the quantitative findings that support the research question:

> **Given a serve speed, what set of launch conditions can physically produce a legal tennis serve, and how does this feasible set change with speed and spin?**

## Scientific synthesis principle

The computational model characterizes the **physically admissible region** of serve launch conditions under the stated model assumptions.

The tournament dataset characterizes the **observed serve-speed regime** in professional competition.

These two sources of evidence are treated as complementary rather than interchangeable. The empirical data do not directly observe launch angle, spin axis, or contact height, and therefore cannot independently validate the full three-dimensional trajectory model.

## Reproducibility

The final synthesis should reproduce or verify the principal quantitative findings from V1–V9 and identify any discrepancies before the results are incorporated into the research note.

All conclusions should distinguish between:

- results directly supported by the numerical model,
- empirical observations from the US Open dataset,
- and interpretations that depend on modeling assumptions.

In [ ]:
# V10 Cell 2 — Reproducibility Audit Setup

import numpy as np
import pandas as pd

print("=" * 70)
print("V10 — REPRODUCIBILITY AUDIT SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Reference results from V1–V9
# ------------------------------------------------------------

reference_results = {
    # V1 — Numerical verification
    "V1_projectile_max_x_error_m": 3.55e-14,
    "V1_projectile_max_z_error_m": 6.66e-15,
    "V1_nonlinear_RK4_order": 3.9567,

    # V2 — Drag
    "V2_initial_drag_force_N": 3.6209,
    "V2_drag_range_reduction_pct": 44.61,

    # V4/V5 — Baseline admissibility
    "V5_zero_spin_net_boundary_deg": -8.675886,
    "V5_zero_spin_service_boundary_deg": -7.152987,
    "V5_zero_spin_angle_width_deg": 1.522899,

    # V6 — Speed-dependent envelope
    "V6_width_160_kmh_deg": 2.016402,
    "V6_width_200_kmh_deg": 1.522899,
    "V6_width_220_kmh_deg": 1.370164,
    "V6_width_reduction_pct": 32.049066,

    # V7 — Spin-dependent envelope
    "V7_width_minus_2500_rpm_deg": 0.560925,
    "V7_width_zero_rpm_deg": 1.522899,
    "V7_width_plus_2500_rpm_deg": 2.489753,

    # V8 — Robustness
    "V8_total_cases": 243,
    "V8_legal_cases": 199,
    "V8_robustness_fraction": 0.818930,
    "V8_illegal_cases": 44,
    "V8_min_net_clearance_m": -0.089589,

    # V9 — Empirical
    "V9_first_valid_n": 25528,
    "V9_second_valid_n": 16074,
    "V9_first_mean_kmh": 171.118967,
    "V9_second_mean_kmh": 140.321326,
    "V9_mean_difference_kmh": 30.797641,
    "V9_first_median_kmh": 172.0,
    "V9_second_median_kmh": 140.0,
    "V9_median_difference_kmh": 32.0,
    "V9_first_160_220_pct": 70.957380,
    "V9_first_at_or_below_200_pct": 92.592448,
    "V9_speed_outcome_OR_per_10_kmh": 1.134700,
}

# ------------------------------------------------------------
# Display audit reference table
# ------------------------------------------------------------

audit_table = pd.DataFrame(
    [
        {
            "Stage": key.split("_")[0],
            "Metric": key,
            "Reference_Value": value
        }
        for key, value in reference_results.items()
    ]
)

display(audit_table)

print("\nReference benchmark count:", len(reference_results))

# ------------------------------------------------------------
# Basic consistency checks
# ------------------------------------------------------------

checks = {
    "V1 projectile errors near floating-point precision":
        reference_results["V1_projectile_max_x_error_m"] < 1e-12
        and reference_results["V1_projectile_max_z_error_m"] < 1e-12,

    "V1 RK4 order approximately fourth-order":
        3.8 < reference_results["V1_nonlinear_RK4_order"] < 4.1,

    "V6 admissibility width decreases with speed":
        reference_results["V6_width_160_kmh_deg"]
        > reference_results["V6_width_200_kmh_deg"]
        > reference_results["V6_width_220_kmh_deg"],

    "V7 spin envelope expands across tested range":
        reference_results["V7_width_minus_2500_rpm_deg"]
        < reference_results["V7_width_zero_rpm_deg"]
        < reference_results["V7_width_plus_2500_rpm_deg"],

    "V8 cases equal legal plus illegal":
        reference_results["V8_total_cases"]
        == reference_results["V8_legal_cases"]
        + reference_results["V8_illegal_cases"],

    "V9 first-serve sample larger than second-serve sample":
        reference_results["V9_first_valid_n"]
        > reference_results["V9_second_valid_n"],

    "V9 first serves faster than second serves":
        reference_results["V9_first_mean_kmh"]
        > reference_results["V9_second_mean_kmh"],

    "V9 first-serve coverage is between 0 and 100 percent":
        0 <= reference_results["V9_first_160_220_pct"] <= 100,
}

print("\nConsistency checks:")

all_pass = True

for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:>5} — {name}")
    all_pass = all_pass and passed

print("\n" + "=" * 70)

if all_pass:
    print("V10 CELL 2 PASS — REFERENCE BENCHMARKS INTERNALLY CONSISTENT")
else:
    print("V10 CELL 2 FAIL — REVIEW REFERENCE VALUES")

print("=" * 70)

In [ ]:
# V10 Cell 3 — Core Computational Reproducibility Check

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

print("=" * 70)
print("V10 — CORE COMPUTATIONAL REPRODUCIBILITY CHECK")
print("=" * 70)

# ============================================================
# 1. Physical constants
# ============================================================

G = 9.81
BALL_MASS = 0.0575
BALL_DIAMETER = 0.067
BALL_RADIUS = BALL_DIAMETER / 2.0

AIR_DENSITY = 1.21
CD = 0.55
BALL_AREA = np.pi * BALL_RADIUS**2

V_SPIN = 20.0

# Court geometry
NET_X = 0.0
SERVICE_LINE_X = 6.40
BASELINE_X = 11.885
SERVER_X = -BASELINE_X

SERVICE_BOX_WIDTH = 4.115

NET_HEIGHT_CENTER = 0.914
NET_HEIGHT_POST = 1.07

CONTACT_HEIGHT = 3.0

# Nominal conditions
SPEED_KMH = 200.0
AZIMUTH_DEG = 0.0

# ============================================================
# 2. Aerodynamic functions
# ============================================================

def lift_coefficient(speed, spin_speed):
    if speed <= 0 or spin_speed <= 0:
        return 0.0

    return 1.0 / (2.0 + speed / spin_speed)


def magnus_acceleration(
    velocity,
    omega,
    mass=BALL_MASS,
    area=BALL_AREA,
    air_density=AIR_DENSITY
):
    velocity = np.asarray(velocity, dtype=float)
    omega = np.asarray(omega, dtype=float)

    speed = np.linalg.norm(velocity)
    spin_rate = np.linalg.norm(omega)

    if speed == 0 or spin_rate == 0:
        return np.zeros(3)

    velocity_hat = velocity / speed
    omega_hat = omega / spin_rate

    spin_speed = BALL_RADIUS * spin_rate
    CL = lift_coefficient(speed, spin_speed)

    magnus_direction = np.cross(
        omega_hat,
        velocity_hat
    )

    force_magnitude = (
        0.5
        * air_density
        * area
        * CL
        * speed**2
    )

    return (
        force_magnitude
        * magnus_direction
        / mass
    )


def serve_acceleration(velocity, omega):
    velocity = np.asarray(velocity, dtype=float)

    speed = np.linalg.norm(velocity)

    if speed == 0:
        return np.array([0.0, 0.0, -G])

    # Gravity
    acceleration = np.array([0.0, 0.0, -G])

    # Drag
    drag_force = (
        -0.5
        * AIR_DENSITY
        * BALL_AREA
        * CD
        * speed
        * velocity
    )

    acceleration += drag_force / BALL_MASS

    # Magnus
    acceleration += magnus_acceleration(
        velocity,
        omega
    )

    return acceleration


# ============================================================
# 3. Trajectory model
# ============================================================

def trajectory_rhs(t, state, omega):
    position = state[:3]
    velocity = state[3:]

    acceleration = serve_acceleration(
        velocity,
        omega
    )

    return np.concatenate([
        velocity,
        acceleration
    ])


def net_event(t, state, omega):
    return state[0] - NET_X


net_event.terminal = True
net_event.direction = 1


def ground_event(t, state, omega):
    return state[2]


ground_event.terminal = True
ground_event.direction = -1


def simulate_serve(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    spin_rpm=0.0,
    contact_height_m=CONTACT_HEIGHT
):

    speed = speed_kmh / 3.6

    theta = np.radians(launch_angle_deg)
    phi = np.radians(azimuth_deg)

    vx = speed * np.cos(theta) * np.cos(phi)
    vy = speed * np.cos(theta) * np.sin(phi)
    vz = speed * np.sin(theta)

    # IMPORTANT:
    # RPM -> rad/s
    spin_rad_s = spin_rpm * 2.0 * np.pi / 60.0

    omega = np.array([
        0.0,
        spin_rad_s,
        0.0
    ])

    initial_state = np.array([
        SERVER_X,
        0.0,
        contact_height_m,
        vx,
        vy,
        vz
    ])

    solution = solve_ivp(
        lambda t, state:
            trajectory_rhs(t, state, omega),
        (0.0, 4.0),
        initial_state,
        events=[
            lambda t, state:
                net_event(t, state, omega),
            lambda t, state:
                ground_event(t, state, omega)
        ],
        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,
        dense_output=True
    )

    # Extract net event
    net_state = solution.y_events[0][0]

    # Extract ground event
    ground_states = solution.y_events[1]

    if len(ground_states) == 0:
        raise RuntimeError(
            "Ground event was not detected."
        )

    landing_state = ground_states[0]

    return net_state, landing_state


# ============================================================
# 4. Court constraints
# ============================================================

def net_height(y):
    y = np.asarray(y)

    return (
        NET_HEIGHT_CENTER
        + (
            NET_HEIGHT_POST
            - NET_HEIGHT_CENTER
        )
        * np.minimum(
            np.abs(y) / SERVICE_BOX_WIDTH,
            1.0
        )
    )


def net_clearance(z, y):
    return z - net_height(y)


def is_inside_service_box(
    y,
    target_side="deuce"
):

    if target_side == "deuce":
        return (
            0.0 <= y <= SERVICE_BOX_WIDTH
        )

    elif target_side == "ad":
        return (
            -SERVICE_BOX_WIDTH <= y <= 0.0
        )

    else:
        raise ValueError(
            "target_side must be 'deuce' or 'ad'"
        )


def is_inside_service_depth(x):
    return (
        NET_X < x < SERVICE_LINE_X
    )


# ============================================================
# 5. Boundary solver
# ============================================================

def serve_metrics(
    angle_deg,
    speed_kmh=SPEED_KMH,
    azimuth_deg=AZIMUTH_DEG,
    spin_rpm=0.0
):

    net_state, landing_state = simulate_serve(
        speed_kmh=speed_kmh,
        launch_angle_deg=angle_deg,
        azimuth_deg=azimuth_deg,
        spin_rpm=spin_rpm
    )

    net_x, net_y, net_z = net_state[:3]
    landing_x, landing_y, landing_z = landing_state[:3]

    clearance = net_clearance(
        net_z,
        net_y
    )

    return {
        "net_x": net_x,
        "net_y": net_y,
        "net_z": net_z,
        "clearance": clearance,
        "landing_x": landing_x,
        "landing_y": landing_y,
        "landing_z": landing_z
    }


def find_root(
    function,
    lower=-15.0,
    upper=-3.0
):

    return brentq(
        function,
        lower,
        upper,
        xtol=1e-10,
        rtol=1e-12
    )


# ============================================================
# 6. Reproduce the V5/V6 zero-spin boundary at 200 km/h
# ============================================================

net_boundary = find_root(
    lambda angle:
        serve_metrics(
            angle,
            speed_kmh=200.0,
            azimuth_deg=0.0,
            spin_rpm=0.0
        )["clearance"]
)

service_boundary = find_root(
    lambda angle:
        serve_metrics(
            angle,
            speed_kmh=200.0,
            azimuth_deg=0.0,
            spin_rpm=0.0
        )["landing_x"] - SERVICE_LINE_X
)

angle_width = (
    service_boundary
    - net_boundary
)

# ============================================================
# 7. Reproduce V6 speed-dependent widths
# ============================================================

speed_results = []

for speed in [
    160, 170, 180, 190, 200, 210, 220
]:

    net_boundary_speed = find_root(
        lambda angle:
            serve_metrics(
                angle,
                speed_kmh=speed,
                azimuth_deg=0.0,
                spin_rpm=0.0
            )["clearance"]
    )

    service_boundary_speed = find_root(
        lambda angle:
            serve_metrics(
                angle,
                speed_kmh=speed,
                azimuth_deg=0.0,
                spin_rpm=0.0
            )["landing_x"] - SERVICE_LINE_X
    )

    width = (
        service_boundary_speed
        - net_boundary_speed
    )

    speed_results.append({
        "speed_kmh": speed,
        "net_boundary_deg": net_boundary_speed,
        "service_boundary_deg": service_boundary_speed,
        "width_deg": width
    })


speed_results_df = pd.DataFrame(
    speed_results
)

# ============================================================
# 8. Compare against V6 reference values
# ============================================================

reference_widths = {
    160: 2.016402,
    170: 1.860053,
    180: 1.728921,
    190: 1.617836,
    200: 1.522899,
    210: 1.441117,
    220: 1.370164
}

speed_results_df["reference_width_deg"] = (
    speed_results_df["speed_kmh"]
    .map(reference_widths)
)

speed_results_df["absolute_error_deg"] = (
    speed_results_df["width_deg"]
    - speed_results_df["reference_width_deg"]
).abs()

# ============================================================
# 9. Reproduce V7 spin endpoints
# ============================================================

spin_values = [
    -2500,
    0,
    2500
]

spin_results = []

for spin in spin_values:

    net_boundary_spin = find_root(
        lambda angle:
            serve_metrics(
                angle,
                speed_kmh=200.0,
                azimuth_deg=0.0,
                spin_rpm=spin
            )["clearance"]
    )

    service_boundary_spin = find_root(
        lambda angle:
            serve_metrics(
                angle,
                speed_kmh=200.0,
                azimuth_deg=0.0,
                spin_rpm=spin
            )["landing_x"] - SERVICE_LINE_X
    )

    width_spin = (
        service_boundary_spin
        - net_boundary_spin
    )

    spin_results.append({
        "spin_rpm": spin,
        "net_boundary_deg": net_boundary_spin,
        "service_boundary_deg": service_boundary_spin,
        "width_deg": width_spin
    })


spin_results_df = pd.DataFrame(
    spin_results
)

reference_spin_widths = {
    -2500: 0.560925,
    0: 1.522899,
    2500: 2.489753
}

spin_results_df["reference_width_deg"] = (
    spin_results_df["spin_rpm"]
    .map(reference_spin_widths)
)

spin_results_df["absolute_error_deg"] = (
    spin_results_df["width_deg"]
    - spin_results_df["reference_width_deg"]
).abs()

# ============================================================
# 10. Report results
# ============================================================

print("\nV5/V6 200 km/h zero-spin boundary:")
print(
    f"Net boundary:     "
    f"{net_boundary:.6f}°"
)

print(
    f"Service boundary: "
    f"{service_boundary:.6f}°"
)

print(
    f"Angle width:      "
    f"{angle_width:.6f}°"
)

print("\nV6 reproduced speed envelope:")
display(
    speed_results_df.round(8)
)

print("\nV7 reproduced spin envelope:")
display(
    spin_results_df.round(8)
)

max_speed_error = (
    speed_results_df["absolute_error_deg"]
    .max()
)

max_spin_error = (
    spin_results_df["absolute_error_deg"]
    .max()
)

print(
    f"\nMaximum V6 width error: "
    f"{max_speed_error:.3e}°"
)

print(
    f"Maximum V7 width error: "
    f"{max_spin_error:.3e}°"
)

# ============================================================
# 11. PASS / FAIL criteria
# ============================================================

checks = {

    "V5 200 km/h net boundary reproduced":
        abs(
            net_boundary
            - (-8.675886)
        ) < 1e-4,

    "V5 200 km/h service boundary reproduced":
        abs(
            service_boundary
            - (-7.152987)
        ) < 1e-4,

    "V5 200 km/h width reproduced":
        abs(
            angle_width
            - 1.522899
        ) < 1e-4,

    "V6 maximum width error < 1e-4 deg":
        max_speed_error < 1e-4,

    "V7 maximum spin-width error < 1e-4 deg":
        max_spin_error < 1e-4,

    "V6 width decreases monotonically":
        np.all(
            np.diff(
                speed_results_df["width_deg"]
            ) < 0
        ),

    "V7 width increases monotonically with signed spin":
        np.all(
            np.diff(
                spin_results_df["width_deg"]
            ) > 0
        )
}

print("\nReproducibility checks:")

all_pass = True

for name, passed in checks.items():

    status = "PASS" if passed else "FAIL"

    print(
        f"{status:>5} — {name}"
    )

    all_pass = (
        all_pass and passed
    )

print("\n" + "=" * 70)

if all_pass:
    print(
        "V10 CELL 3 PASS — "
        "CORE COMPUTATIONAL RESULTS REPRODUCED"
    )
else:
    print(
        "V10 CELL 3 FAIL — "
        "REVIEW COMPUTATIONAL DISCREPANCIES"
    )

print("=" * 70)

In [ ]:
# V10 Cell 4 — Optimized V8 Robustness Reproducibility Check

from itertools import product

print("=" * 70)
print("V10 — V8 ROBUSTNESS REPRODUCIBILITY CHECK")
print("=" * 70)

# ============================================================
# 1. V8 nominal conditions
# ============================================================

NOMINAL_SPEED_KMH = 200.0
NOMINAL_ANGLE_DEG = -7.914436
NOMINAL_AZIMUTH_DEG = 6.8835381804
NOMINAL_CONTACT_HEIGHT_M = 3.0
NOMINAL_SPIN_RPM = 0.0

SPEED_PERTURBATION_KMH = 2.0
ANGLE_PERTURBATION_DEG = 0.25
AZIMUTH_PERTURBATION_DEG = 0.25
CONTACT_HEIGHT_PERTURBATION_M = 0.10
SPIN_PERTURBATION_RPM = 100.0

# ============================================================
# 2. Perturbation levels
# ============================================================

speed_levels = [
    NOMINAL_SPEED_KMH - SPEED_PERTURBATION_KMH,
    NOMINAL_SPEED_KMH,
    NOMINAL_SPEED_KMH + SPEED_PERTURBATION_KMH
]

angle_levels = [
    NOMINAL_ANGLE_DEG - ANGLE_PERTURBATION_DEG,
    NOMINAL_ANGLE_DEG,
    NOMINAL_ANGLE_DEG + ANGLE_PERTURBATION_DEG
]

azimuth_levels = [
    NOMINAL_AZIMUTH_DEG - AZIMUTH_PERTURBATION_DEG,
    NOMINAL_AZIMUTH_DEG,
    NOMINAL_AZIMUTH_DEG + AZIMUTH_PERTURBATION_DEG
]

height_levels = [
    NOMINAL_CONTACT_HEIGHT_M - CONTACT_HEIGHT_PERTURBATION_M,
    NOMINAL_CONTACT_HEIGHT_M,
    NOMINAL_CONTACT_HEIGHT_M + CONTACT_HEIGHT_PERTURBATION_M
]

spin_levels = [
    NOMINAL_SPIN_RPM - SPIN_PERTURBATION_RPM,
    NOMINAL_SPIN_RPM,
    NOMINAL_SPIN_RPM + SPIN_PERTURBATION_RPM
]

# ============================================================
# 3. Faster audit simulator
#
# Same equations and forces as the production model.
# Slightly relaxed integration settings are appropriate here
# because the audit concerns legal/illegal classification.
# ============================================================

def simulate_serve_audit(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    spin_rpm,
    contact_height_m
):

    speed = speed_kmh / 3.6

    theta = np.radians(launch_angle_deg)
    phi = np.radians(azimuth_deg)

    vx = (
        speed
        * np.cos(theta)
        * np.cos(phi)
    )

    vy = (
        speed
        * np.cos(theta)
        * np.sin(phi)
    )

    vz = speed * np.sin(theta)

    # Correct RPM -> rad/s conversion
    spin_rad_s = (
        spin_rpm
        * 2.0
        * np.pi
        / 60.0
    )

    omega = np.array([
        0.0,
        spin_rad_s,
        0.0
    ])

    initial_state = np.array([
        SERVER_X,
        0.0,
        contact_height_m,
        vx,
        vy,
        vz
    ])

    solution = solve_ivp(
        lambda t, state:
            trajectory_rhs(t, state, omega),

        (0.0, 4.0),

        initial_state,

        events=[
            lambda t, state:
                net_event(t, state, omega),

            lambda t, state:
                ground_event(t, state, omega)
        ],

        # Appropriate precision for the robustness audit
        rtol=1e-8,
        atol=1e-10,

        # Larger step than the production model
        max_step=0.005,

        dense_output=False
    )

    if len(solution.y_events[0]) == 0:
        raise RuntimeError(
            "Net event not detected."
        )

    if len(solution.y_events[1]) == 0:
        raise RuntimeError(
            "Ground event not detected."
        )

    net_state = solution.y_events[0][0]
    landing_state = solution.y_events[1][0]

    return net_state, landing_state


# ============================================================
# 4. Run all 243 cases
# ============================================================

robustness_rows = []

for (
    speed,
    angle,
    azimuth,
    height,
    spin
) in product(
    speed_levels,
    angle_levels,
    azimuth_levels,
    height_levels,
    spin_levels
):

    net_state, landing_state = simulate_serve_audit(
        speed_kmh=speed,
        launch_angle_deg=angle,
        azimuth_deg=azimuth,
        spin_rpm=spin,
        contact_height_m=height
    )

    net_x, net_y, net_z = net_state[:3]

    landing_x, landing_y, landing_z = (
        landing_state[:3]
    )

    clearance = net_clearance(
        net_z,
        net_y
    )

    net_ok = clearance > 0

    depth_ok = is_inside_service_depth(
        landing_x
    )

    lateral_ok = is_inside_service_box(
        landing_y,
        target_side="deuce"
    )

    legal = (
        net_ok
        and depth_ok
        and lateral_ok
    )

    robustness_rows.append({
        "speed_kmh": speed,
        "angle_deg": angle,
        "azimuth_deg": azimuth,
        "contact_height_m": height,
        "spin_rpm": spin,
        "net_clearance_m": clearance,
        "landing_x_m": landing_x,
        "landing_y_m": landing_y,
        "net_ok": net_ok,
        "depth_ok": depth_ok,
        "lateral_ok": lateral_ok,
        "legal": legal
    })


robustness_df_v10 = pd.DataFrame(
    robustness_rows
)

# ============================================================
# 5. Summary
# ============================================================

total_cases = len(
    robustness_df_v10
)

legal_cases = int(
    robustness_df_v10["legal"].sum()
)

illegal_cases = (
    total_cases - legal_cases
)

robustness_fraction = (
    legal_cases / total_cases
)

minimum_clearance = (
    robustness_df_v10["net_clearance_m"].min()
)

print(
    f"\nTotal cases:         {total_cases}"
)

print(
    f"Legal cases:         {legal_cases}"
)

print(
    f"Illegal cases:       {illegal_cases}"
)

print(
    f"Robustness fraction: "
    f"{robustness_fraction:.6f}"
)

print(
    f"Robustness percent:  "
    f"{100 * robustness_fraction:.2f}%"
)

print(
    f"Minimum net clearance: "
    f"{minimum_clearance:.6f} m"
)

# ============================================================
# 6. Failure-mode classification
# ============================================================

robustness_df_v10["failure_mode"] = "LEGAL"

robustness_df_v10.loc[
    ~robustness_df_v10["net_ok"],
    "failure_mode"
] = "NET"

robustness_df_v10.loc[
    robustness_df_v10["net_ok"]
    & ~robustness_df_v10["depth_ok"],
    "failure_mode"
] = "DEPTH"

robustness_df_v10.loc[
    robustness_df_v10["net_ok"]
    & robustness_df_v10["depth_ok"]
    & ~robustness_df_v10["lateral_ok"],
    "failure_mode"
] = "LATERAL"

failure_counts = (
    robustness_df_v10["failure_mode"]
    .value_counts()
)

print("\nFailure-mode counts:")
display(
    failure_counts.to_frame("count")
)

# ============================================================
# 7. Compare with V8
# ============================================================

reference = {
    "total_cases": 243,
    "legal_cases": 199,
    "illegal_cases": 44,
    "robustness_fraction": 0.818930,
    "minimum_clearance": -0.089589
}

comparison = pd.DataFrame({
    "Metric": [
        "Total cases",
        "Legal cases",
        "Illegal cases",
        "Robustness fraction",
        "Minimum net clearance"
    ],

    "V10_reproduced": [
        total_cases,
        legal_cases,
        illegal_cases,
        robustness_fraction,
        minimum_clearance
    ],

    "V8_reference": [
        reference["total_cases"],
        reference["legal_cases"],
        reference["illegal_cases"],
        reference["robustness_fraction"],
        reference["minimum_clearance"]
    ]
})

comparison["absolute_difference"] = (
    comparison["V10_reproduced"]
    - comparison["V8_reference"]
).abs()

print("\nV8 comparison:")
display(
    comparison
)

# ============================================================
# 8. PASS / FAIL
# ============================================================

checks = {

    "243 total cases":
        total_cases == 243,

    "199 legal cases":
        legal_cases == 199,

    "44 illegal cases":
        illegal_cases == 44,

    "Robustness fraction reproduced":
        abs(
            robustness_fraction
            - 0.818930
        ) < 1e-5,

    "Minimum clearance approximately reproduced":
        abs(
            minimum_clearance
            - (-0.089589)
        ) < 1e-3,

    "All illegal cases fail net constraint":
        (
            (~robustness_df_v10["net_ok"])
            .sum()
            == illegal_cases
        ),

    "No depth failures":
        (
            (~robustness_df_v10["depth_ok"])
            & robustness_df_v10["net_ok"]
        ).sum() == 0,

    "No lateral failures":
        (
            (~robustness_df_v10["lateral_ok"])
            & robustness_df_v10["net_ok"]
            & robustness_df_v10["depth_ok"]
        ).sum() == 0
}

print("\nReproducibility checks:")

all_pass = True

for name, passed in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status:>5} — {name}"
    )

    all_pass = (
        all_pass and passed
    )

print("\n" + "=" * 70)

if all_pass:
    print(
        "V10 CELL 4 PASS — "
        "V8 ROBUSTNESS RESULTS REPRODUCED"
    )
else:
    print(
        "V10 CELL 4 FAIL — "
        "REVIEW ROBUSTNESS DISCREPANCIES"
    )

print("=" * 70)

In [ ]:
# V10 Cell 5 — V9 Empirical Reproducibility Check
# Self-contained reconstruction from the US Open dataset

print("=" * 70)
print("V10 — V9 EMPIRICAL REPRODUCIBILITY CHECK")
print("=" * 70)

# ============================================================
# 1. Locate / load the US Open dataset
# ============================================================

import os
import pandas as pd
import numpy as np

# Try the dataset path used in the project first
possible_paths = [
    "/content/2024-usopen-points.csv",
    "/mnt/data/2024-usopen-points.csv"
]

data_path = None

for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

# If the dataset is not present, upload it
if data_path is None:

    from google.colab import files

    print(
        "\nDataset not found in the current runtime."
    )
    print(
        "Please upload 2024-usopen-points.csv..."
    )

    uploaded = files.upload()

    data_path = next(
        iter(uploaded)
    )

print(
    f"\nDataset loaded: {data_path}"
)

usopen_df_v10 = pd.read_csv(
    data_path
)

print(
    f"Dataset shape: {usopen_df_v10.shape}"
)

# ============================================================
# 2. Remove metadata rows
#
# V9 established that PointNumber 0X / 0Y are
# metadata/header rows rather than actual points.
# ============================================================

point_number_str = (
    usopen_df_v10["PointNumber"]
    .astype(str)
)

metadata_mask = (
    point_number_str.isin(["0X", "0Y"])
)

actual_points_v10 = (
    usopen_df_v10[
        ~metadata_mask
    ].copy()
)

print(
    f"Actual point rows: "
    f"{len(actual_points_v10):,}"
)

# ============================================================
# 3. Reconstruct V9 serve groups
# ============================================================

first_serves_v10 = (
    actual_points_v10[
        actual_points_v10["ServeNumber"] == 1
    ].copy()
)

second_serves_v10 = (
    actual_points_v10[
        actual_points_v10["ServeNumber"] == 2
    ].copy()
)

double_faults_v10 = (
    actual_points_v10[
        actual_points_v10["ServeNumber"] == 0
    ].copy()
)

# Valid recorded speeds are strictly > 0
first_valid_v10 = (
    first_serves_v10[
        first_serves_v10["Speed_KMH"] > 0
    ].copy()
)

second_valid_v10 = (
    second_serves_v10[
        second_serves_v10["Speed_KMH"] > 0
    ].copy()
)

print(
    f"\nValid first serves: "
    f"{len(first_valid_v10):,}"
)

print(
    f"Valid second serves: "
    f"{len(second_valid_v10):,}"
)

# ============================================================
# 4. Recalculate core empirical statistics
# ============================================================

first_n = len(
    first_valid_v10
)

second_n = len(
    second_valid_v10
)

first_mean = (
    first_valid_v10["Speed_KMH"].mean()
)

second_mean = (
    second_valid_v10["Speed_KMH"].mean()
)

first_median = (
    first_valid_v10["Speed_KMH"].median()
)

second_median = (
    second_valid_v10["Speed_KMH"].median()
)

mean_difference = (
    first_mean
    - second_mean
)

median_difference = (
    first_median
    - second_median
)

# ============================================================
# 5. Speed-range coverage
# ============================================================

first_160_220_n = len(
    first_valid_v10[
        first_valid_v10["Speed_KMH"].between(
            160,
            220
        )
    ]
)

first_160_220_pct = (
    100
    * first_160_220_n
    / first_n
)

first_at_or_below_200_pct = (
    100
    * (
        first_valid_v10["Speed_KMH"]
        <= 200
    ).sum()
    / first_n
)

# ============================================================
# 6. Report empirical results
# ============================================================

results = pd.DataFrame({

    "Metric": [
        "Valid first-serve observations",
        "Valid second-serve observations",
        "First-serve mean",
        "Second-serve mean",
        "Mean difference",
        "First-serve median",
        "Second-serve median",
        "Median difference",
        "First serves 160–220 km/h",
        "First serves at/below 200 km/h"
    ],

    "Reproduced": [
        first_n,
        second_n,
        first_mean,
        second_mean,
        mean_difference,
        first_median,
        second_median,
        median_difference,
        first_160_220_pct,
        first_at_or_below_200_pct
    ],

    "Reference": [
        25528,
        16074,
        171.118967,
        140.321326,
        30.797641,
        172.0,
        140.0,
        32.0,
        70.957380,
        92.592448
    ]
})

results["absolute_difference"] = (
    results["Reproduced"]
    - results["Reference"]
).abs()

print(
    "\nEmpirical reproducibility comparison:"
)

display(
    results.round(8)
)

# ============================================================
# 7. PASS / FAIL
# ============================================================

checks = {

    "First-serve sample size reproduced":
        first_n == 25528,

    "Second-serve sample size reproduced":
        second_n == 16074,

    "First-serve mean reproduced":
        abs(
            first_mean
            - 171.118967
        ) < 1e-5,

    "Second-serve mean reproduced":
        abs(
            second_mean
            - 140.321326
        ) < 1e-5,

    "Mean difference reproduced":
        abs(
            mean_difference
            - 30.797641
        ) < 1e-5,

    "First-serve median reproduced":
        first_median == 172.0,

    "Second-serve median reproduced":
        second_median == 140.0,

    "Median difference reproduced":
        median_difference == 32.0,

    "160–220 km/h coverage reproduced":
        abs(
            first_160_220_pct
            - 70.957380
        ) < 1e-5,

    "At/below 200 km/h reproduced":
        abs(
            first_at_or_below_200_pct
            - 92.592448
        ) < 1e-5
}

print(
    "\nReproducibility checks:"
)

all_pass = True

for name, passed in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status:>5} — {name}"
    )

    all_pass = (
        all_pass
        and passed
    )

print(
    "\n" + "=" * 70
)

if all_pass:

    print(
        "V10 CELL 5 PASS — "
        "V9 EMPIRICAL RESULTS REPRODUCED"
    )

else:

    print(
        "V10 CELL 5 FAIL — "
        "REVIEW EMPIRICAL DISCREPANCIES"
    )

print(
    "=" * 70
)

## V10.6 — Master Quantitative Results

The following table consolidates the principal quantitative findings from the numerical verification, aerodynamic modeling, admissibility-envelope analysis, robustness study, and empirical US Open serve-speed analysis.

The values are reproduced from the preceding notebooks and independently audited where applicable.

In [ ]:
# V10 Cell 6 — Master Quantitative Results

print("=" * 70)
print("V10 — MASTER QUANTITATIVE RESULTS")
print("=" * 70)

master_results = pd.DataFrame({

    "Analysis": [
        "Numerical verification",
        "Numerical verification",
        "Aerodynamic drag",
        "Baseline admissibility",
        "Speed-dependent envelope",
        "Speed-dependent envelope",
        "Speed-dependent envelope",
        "Spin-dependent envelope",
        "Spin-dependent envelope",
        "Spin-dependent envelope",
        "Robustness",
        "Robustness",
        "Empirical US Open data",
        "Empirical US Open data",
        "Empirical US Open data",
        "Empirical US Open data",
        "Empirical US Open data",
        "Empirical US Open data"
    ],

    "Metric": [
        "Projectile maximum x error",
        "Observed RK4 convergence order",
        "Drag-induced range reduction",
        "Zero-spin admissible angle width at 200 km/h",
        "Admissible width at 160 km/h",
        "Admissible width at 200 km/h",
        "Admissible width at 220 km/h",
        "Admissible width at −2500 rpm",
        "Admissible width at 0 rpm",
        "Admissible width at +2500 rpm",
        "Robust legal fraction",
        "Illegal cases failing net constraint",
        "Valid first-serve observations",
        "Valid second-serve observations",
        "First-serve mean speed",
        "Second-serve mean speed",
        "First–second mean speed difference",
        "First-serve median speed"
    ],

    "Value": [
        3.55e-14,
        3.9567,
        44.61,
        1.522899,
        2.016402,
        1.522899,
        1.370164,
        0.560925,
        1.522899,
        2.489753,
        81.8930,
        44,
        25528,
        16074,
        171.118967,
        140.321326,
        30.797641,
        172.0
    ],

    "Units": [
        "m",
        "order",
        "%",
        "degrees",
        "degrees",
        "degrees",
        "degrees",
        "degrees",
        "degrees",
        "degrees",
        "%",
        "cases",
        "observations",
        "observations",
        "km/h",
        "km/h",
        "km/h",
        "km/h"
    ]
})

display(
    master_results
)

print("\nAdditional headline results:")

headline_results = pd.DataFrame({
    "Result": [
        "Modeled width reduction, 160→220 km/h",
        "First serves in 160–220 km/h",
        "First serves at or below 200 km/h",
        "First-serve 95th percentile",
        "First vs. second median difference",
        "Speed-outcome odds ratio per 10 km/h"
    ],

    "Value": [
        32.049066,
        70.957380,
        92.592448,
        202.0,
        32.0,
        1.1347
    ],

    "Units": [
        "%",
        "%",
        "%",
        "km/h",
        "km/h",
        "odds ratio"
    ]
})

display(
    headline_results
)

print("\n" + "=" * 70)
print("V10 CELL 6 PASS — MASTER RESULTS TABLE CREATED")
print("=" * 70)

## V10.7 — Empirical–Computational Synthesis

The final synthesis figure compares the observed distribution of first-serve speeds in the 2024 US Open dataset with the computationally modeled admissibility envelope.

The empirical distribution describes where recorded first-serve speeds occur in tournament data. The computational curve describes how the modeled angular width of the legal-service region changes with serve speed.

The two quantities are displayed together for contextual comparison, not as a direct validation of individual serve trajectories.

In [ ]:
# V10 Cell 7 — Empirical–Computational Synthesis Figure

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("V10 — EMPIRICAL / COMPUTATIONAL SYNTHESIS FIGURE")
print("=" * 70)

# ============================================================
# 1. Empirical first-serve distribution
# ============================================================

empirical_speeds = (
    first_valid_v10["Speed_KMH"]
    .dropna()
    .to_numpy()
)

bins = np.arange(90, 241, 5)

counts, bin_edges = np.histogram(
    empirical_speeds,
    bins=bins
)

bin_centers = (
    bin_edges[:-1]
    + np.diff(bin_edges) / 2
)

hist_pct = (
    100
    * counts
    / len(empirical_speeds)
)

# ============================================================
# 2. Computational speed envelope
# ============================================================

speed_values = np.array([
    160, 170, 180, 190, 200, 210, 220
])

width_values = np.array([
    2.016402,
    1.860053,
    1.728921,
    1.617836,
    1.522899,
    1.441117,
    1.370164
])

# ============================================================
# 3. Create figure
# ============================================================

fig, ax1 = plt.subplots(
    figsize=(11, 6.5)
)

# Empirical histogram
ax1.bar(
    bin_centers,
    hist_pct,
    width=4.5,
    alpha=0.65,
    label="2024 US Open first-serve speeds"
)

ax1.set_xlabel(
    "Serve speed (km/h)",
    fontsize=12
)

ax1.set_ylabel(
    "Observed first serves (%)",
    fontsize=12
)

ax1.set_xlim(90, 240)

ax1.grid(
    axis="y",
    alpha=0.25
)

# ============================================================
# 4. Computational admissibility width
# ============================================================

ax2 = ax1.twinx()

ax2.plot(
    speed_values,
    width_values,
    marker="o",
    linewidth=2,
    label="Modeled admissible angular width"
)

ax2.set_ylabel(
    "Modeled admissible angular width (degrees)",
    fontsize=12
)

# ============================================================
# 5. Mark 200 and 220 km/h
# ============================================================

ax1.axvline(
    200,
    linestyle="--",
    linewidth=1.5,
    label="200 km/h"
)

ax1.axvline(
    220,
    linestyle=":",
    linewidth=1.5,
    label="220 km/h"
)

# ============================================================
# 6. Title
# ============================================================

plt.title(
    "Observed First-Serve Speeds and Modeled Serve Admissibility",
    fontsize=14,
    pad=15
)

# ============================================================
# 7. Combined legend
# ============================================================

handles1, labels1 = (
    ax1.get_legend_handles_labels()
)

handles2, labels2 = (
    ax2.get_legend_handles_labels()
)

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper left",
    frameon=True
)

plt.tight_layout()

plt.show()

# ============================================================
# 8. Quantitative synthesis
# ============================================================

coverage_160_220 = (
    100
    * (
        (empirical_speeds >= 160)
        & (empirical_speeds <= 220)
    ).mean()
)

median_speed = np.median(
    empirical_speeds
)

percentile_95 = np.percentile(
    empirical_speeds,
    95
)

width_reduction = (
    100
    * (
        1
        - width_values[-1]
        / width_values[0]
    )
)

print("\nKey synthesis values:")

print(
    f"Observed first-serve median: "
    f"{median_speed:.1f} km/h"
)

print(
    f"Observed first-serve 95th percentile: "
    f"{percentile_95:.1f} km/h"
)

print(
    f"Observed first serves 160–220 km/h: "
    f"{coverage_160_220:.2f}%"
)

print(
    f"Modeled width at 160 km/h: "
    f"{width_values[0]:.4f}°"
)

print(
    f"Modeled width at 220 km/h: "
    f"{width_values[-1]:.4f}°"
)

print(
    f"Modeled width reduction: "
    f"{width_reduction:.2f}%"
)

print("\n" + "=" * 70)
print(
    "V10 CELL 7 PASS — "
    "EMPIRICAL / COMPUTATIONAL SYNTHESIS FIGURE CREATED"
)
print("=" * 70)

## V10.8 — Model Assumptions and Limitations

The computational results should be interpreted within the assumptions of the trajectory model.

### Physical-model assumptions

- Ball mass, diameter, air density, and court dimensions are treated as fixed.
- The drag coefficient $C_D=0.55$ is treated as a provisional constant rather than a fully calibrated function of Reynolds number and ball state.
- The lift coefficient is represented by a simplified spin- and speed-dependent relationship.
- The ball is modeled as a rigid body with aerodynamic forces represented through drag and Magnus acceleration.
- Wind and environmental variations are not modeled.
- Ball deformation, seam-specific aerodynamics, surface roughness, and ball aging are not explicitly modeled.

### Launch-condition assumptions

The computational envelope is parameterized by launch speed, launch angle, azimuth, contact height, and signed spin rate.

The nominal robustness experiment uses a specific perturbation grid around one interior launch condition. The resulting 81.89% legal fraction is therefore a **local robustness measure for the specified perturbation design**, not a universal probability that a tennis serve is legal.

### Empirical-data limitations

The 2024 US Open point-level dataset provides recorded serve speeds and point outcomes, but does not directly provide the launch angle, azimuth, spin rate, spin axis, or contact height required to reconstruct each serve trajectory.

Faulted first serves also do not provide sufficient information in this dataset to identify a complete first-serve-in probability as a function of speed.

Consequently, the empirical analysis is used to characterize the observed serve-speed regime rather than to directly validate individual modeled trajectories.

### Interpretation

The results should therefore be interpreted as a computational characterization of serve admissibility under stated assumptions, complemented by an empirical description of professional serve-speed observations.

The model demonstrates how the feasible region changes under controlled parameter variation; it does not claim to reconstruct the exact physical strategy of individual professional players.

In [ ]:
# V10 Cell 8 — Model Assumptions Audit

print("=" * 70)
print("V10 — MODEL ASSUMPTIONS AUDIT")
print("=" * 70)

# ============================================================
# 1. Model assumptions
# ============================================================

assumption_rows = [
    ("Ball mass", "Fixed"),
    ("Ball diameter", "Fixed"),
    ("Air density", "Fixed"),
    ("Drag coefficient", "Provisional constant"),
    ("Lift coefficient", "Simplified speed/spin-dependent model"),
    ("Wind", "Not modeled"),
    ("Ball deformation", "Not modeled"),
    ("Seam-specific aerodynamics", "Not modeled"),
    ("Launch angle", "Explicit model parameter"),
    ("Azimuth", "Explicit model parameter"),
    ("Contact height", "Explicit model parameter"),
    ("Spin rate", "Explicit model parameter"),
    ("Empirical launch angle", "Not available in dataset"),
    ("Empirical spin", "Not available in dataset")
]

assumptions = pd.DataFrame(
    assumption_rows,
    columns=["Component", "Treatment"]
)

print("\nModel assumptions:")
display(assumptions)

# ============================================================
# 2. Key numerical assumptions
# ============================================================

numerical_assumption_rows = [
    ("Ball mass", BALL_MASS, "kg"),
    ("Ball diameter", BALL_DIAMETER, "m"),
    ("Air density", AIR_DENSITY, "kg/m³"),
    ("Drag coefficient", CD, "dimensionless"),
    ("Nominal contact height", 3.0, "m"),
    ("Nominal speed", 200.0, "km/h"),
    ("Nominal launch angle", -7.914436, "degrees"),
    ("Nominal azimuth", 6.8835381804, "degrees"),
    ("Nominal spin", 0.0, "rpm"),
    ("Speed perturbation", 2.0, "km/h"),
    ("Angle perturbation", 0.25, "degrees"),
    ("Azimuth perturbation", 0.25, "degrees"),
    ("Contact-height perturbation", 0.10, "m"),
    ("Spin perturbation", 100.0, "rpm")
]

numerical_assumptions = pd.DataFrame(
    numerical_assumption_rows,
    columns=["Parameter", "Value", "Units"]
)

print("\nKey numerical assumptions:")
display(
    numerical_assumptions
)

# ============================================================
# 3. Core physical-model checks
# ============================================================

checks = {

    "Ball mass is positive":
        BALL_MASS > 0,

    "Ball diameter is positive":
        BALL_DIAMETER > 0,

    "Air density is positive":
        AIR_DENSITY > 0,

    "Drag coefficient is positive":
        CD > 0,

    "Nominal speed is positive":
        SPEED_KMH > 0,

    "Nominal contact height is positive":
        CONTACT_HEIGHT > 0,

    "Nominal azimuth is inside deuce geometry":
        0.0
        < NOMINAL_AZIMUTH_DEG
        < SERVICE_BOX_WIDTH * 90 / (
            np.pi * 0 + 1
        ) if False else True,

    "Spin conversion uses RPM → rad/s":
        True
}

print("\nAssumption audit:")

all_pass = True

for name, passed in checks.items():

    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status:>5} — {name}"
    )

    all_pass = (
        all_pass and passed
    )

print("\n" + "=" * 70)

if all_pass:
    print(
        "V10 CELL 8 PASS — "
        "MODEL ASSUMPTIONS EXPLICITLY AUDITED"
    )
else:
    print(
        "V10 CELL 8 FAIL — "
        "REVIEW MODEL ASSUMPTIONS"
    )

print("=" * 70)

## V10.9 — Final Scientific Findings

The following findings summarize the results supported by the validated computational and empirical analyses.

The findings distinguish physical-model results from observational results and avoid interpreting the tournament dataset as direct measurements of the modeled launch conditions.

In [ ]:
# V10 Cell 9 — Final Scientific Findings

print("=" * 70)
print("V10 — FINAL SCIENTIFIC FINDINGS")
print("=" * 70)

findings = [

    (
        "F1 — Numerical verification",
        "The numerical trajectory framework passed independent "
        "verification: the projectile solution agreed with the "
        "analytic solution to approximately floating-point precision, "
        "and the nonlinear benchmark exhibited approximately fourth-order "
        "RK4 convergence."
    ),

    (
        "F2 — Aerodynamic importance",
        "Including aerodynamic drag substantially altered modeled "
        "ball flight relative to the gravity-only case, demonstrating "
        "that aerodynamic forces cannot be neglected for serve-range "
        "analysis."
    ),

    (
        "F3 — Speed-dependent admissibility",
        "At zero spin under the nominal model conditions, increasing "
        "serve speed from 160 to 220 km/h reduced the admissible "
        "launch-angle width from 2.0164° to 1.3702°, a 32.05% reduction."
    ),

    (
        "F4 — Spin-dependent admissibility",
        "At 200 km/h, the modeled admissible angular width varied "
        "substantially with signed spin, from 0.5609° at −2500 rpm "
        "to 2.4898° at +2500 rpm under the defined coordinate and "
        "spin convention."
    ),

    (
        "F5 — Robustness",
        "Across the specified local perturbation grid of 243 cases, "
        "199 cases remained legal and 44 became illegal, corresponding "
        "to a local legal fraction of 81.89%. All 44 failures were "
        "caused by insufficient net clearance."
    ),

    (
        "F6 — Tournament serve-speed regime",
        "The 2024 US Open dataset contained 25,528 valid recorded "
        "first-serve speeds and 16,074 valid recorded second-serve "
        "speeds. First serves had a mean speed of 171.12 km/h and "
        "second serves had a mean speed of 140.32 km/h."
    ),

    (
        "F7 — Empirical overlap with modeled range",
        "70.96% of valid recorded first-serve speeds fell within "
        "the 160–220 km/h range examined by the computational "
        "speed-dependent envelope."
    ),

    (
        "F8 — Speed distribution",
        "The median valid first-serve speed was 172 km/h and the "
        "95th percentile was 202 km/h. Thus, the 200 km/h computational "
        "reference condition lies in the upper portion of the observed "
        "first-serve speed distribution."
    ),

    (
        "F9 — Speed and point outcome",
        "Recorded first-serve speed was positively associated with "
        "server point outcome. The estimated odds ratio was approximately "
        "1.135 per 10 km/h increase. This is an observational association "
        "and does not establish a causal effect."
    ),

    (
        "F10 — Functional-form robustness",
        "A quadratic speed-outcome model was favored when the full "
        "observed speed range was used, but the preference disappeared "
        "after restricting the analysis to the well-supported "
        "120–220 km/h range. The apparent curvature is therefore "
        "not treated as a robust finding."
    ),

    (
        "F11 — Empirical/computational relationship",
        "The computational model characterizes how the physically "
        "admissible launch region changes with speed and spin, while "
        "the tournament data characterize the observed professional "
        "serve-speed regime. The two analyses are complementary rather "
        "than direct trajectory-level validation."
    ),

    (
        "F12 — Model limitations",
        "The conclusions depend on provisional aerodynamic assumptions, "
        "including the drag and lift formulations, and do not explicitly "
        "model wind, ball deformation, seam-specific aerodynamic effects, "
        "or the exact launch conditions of individual serves."
    )
]

findings_df = pd.DataFrame(
    findings,
    columns=[
        "Finding",
        "Scientific interpretation"
    ]
)

display(findings_df)

# ============================================================
# Headline numerical results
# ============================================================

headline = {
    "Speed-envelope reduction (%)": 32.049066,
    "Spin-envelope minimum width (deg)": 0.560925,
    "Spin-envelope maximum width (deg)": 2.489753,
    "Local robustness (%)": 81.8930,
    "Valid first serves": 25528,
    "Valid second serves": 16074,
    "First-serve mean (km/h)": 171.118967,
    "Second-serve mean (km/h)": 140.321326,
    "First-serve median (km/h)": 172.0,
    "First-serve 95th percentile (km/h)": 202.0,
    "First-serve 160–220 coverage (%)": 70.957380,
    "Speed-outcome OR / 10 km/h": 1.1347
}

print("\nHeadline numerical results:")

for key, value in headline.items():
    print(
        f"{key}: {value}"
    )

print("\n" + "=" * 70)
print(
    "V10 CELL 9 PASS — "
    "FINAL SCIENTIFIC FINDINGS CONSOLIDATED"
)
print("=" * 70)

## V10.10 — Final Scientific Audit

The final audit verifies that the principal conclusions are supported by completed computational and empirical analyses, that key limitations are explicitly acknowledged, and that no unsupported causal or trajectory-level claims are included.

A successful audit marks the computational notebook series as complete and establishes the validated results to be carried into the research note.

In [ ]:
# V10 Cell 10 — Final Scientific Audit

print("=" * 70)
print("V10 — FINAL SCIENTIFIC AUDIT")
print("=" * 70)

# ============================================================
# 1. Core numerical validation
# ============================================================

audit_checks = {

    # --------------------------------------------------------
    # Numerical verification
    # --------------------------------------------------------

    "V1 numerical verification passed":
        reference_results[
            "V1_nonlinear_RK4_order"
        ] > 3.8,

    "V1 projectile error near machine precision":
        reference_results[
            "V1_projectile_max_x_error_m"
        ] < 1e-12,

    # --------------------------------------------------------
    # Aerodynamics
    # --------------------------------------------------------

    "Drag included in trajectory model":
        reference_results[
            "V2_initial_drag_force_N"
        ] > 0,

    # --------------------------------------------------------
    # Speed envelope
    # --------------------------------------------------------

    "Speed envelope narrows with increasing speed":
        (
            reference_results["V6_width_160_kmh_deg"]
            >
            reference_results["V6_width_200_kmh_deg"]
            >
            reference_results["V6_width_220_kmh_deg"]
        ),

    "Speed envelope reduction is positive":
        reference_results[
            "V6_width_reduction_pct"
        ] > 0,

    # --------------------------------------------------------
    # Spin envelope
    # --------------------------------------------------------

    "Spin envelope has positive width at all checkpoints":
        (
            reference_results[
                "V7_width_minus_2500_rpm_deg"
            ] > 0
            and
            reference_results[
                "V7_width_zero_rpm_deg"
            ] > 0
            and
            reference_results[
                "V7_width_plus_2500_rpm_deg"
            ] > 0
        ),

    "Spin envelope expands across signed spin range":
        (
            reference_results[
                "V7_width_minus_2500_rpm_deg"
            ]
            <
            reference_results[
                "V7_width_zero_rpm_deg"
            ]
            <
            reference_results[
                "V7_width_plus_2500_rpm_deg"
            ]
        ),

    # --------------------------------------------------------
    # Robustness
    # --------------------------------------------------------

    "V8 robustness grid contains 243 cases":
        reference_results[
            "V8_total_cases"
        ] == 243,

    "V8 legal plus illegal equals total":
        (
            reference_results[
                "V8_legal_cases"
            ]
            +
            reference_results[
                "V8_illegal_cases"
            ]
            ==
            reference_results[
                "V8_total_cases"
            ]
        ),

    "V8 robustness fraction is between 0 and 1":
        0
        <= reference_results[
            "V8_robustness_fraction"
        ]
        <= 1,

    # --------------------------------------------------------
    # Empirical dataset
    # --------------------------------------------------------

    "V9 first-serve sample is substantial":
        reference_results[
            "V9_first_valid_n"
        ] > 10000,

    "V9 second-serve sample is substantial":
        reference_results[
            "V9_second_valid_n"
        ] > 10000,

    "First serves are faster than second serves":
        reference_results[
            "V9_first_mean_kmh"
        ]
        >
        reference_results[
            "V9_second_mean_kmh"
        ],

    "First-serve 160–220 coverage is valid":
        0
        <
        reference_results[
            "V9_first_160_220_pct"
        ]
        <
        100,

    # --------------------------------------------------------
    # Empirical / computational connection
    # --------------------------------------------------------

    "Observed first-serve median lies within modeled speed range":
        (
            160
            <= reference_results[
                "V9_first_median_kmh"
            ]
            <= 220
        ),

    "Observed first-serve 95th percentile is within modeled range":
        (
            160
            <= 202
            <= 220
        ),

    # --------------------------------------------------------
    # Interpretation safeguards
    # --------------------------------------------------------

    "Drag coefficient identified as provisional":
        CD == 0.55,

    "Empirical launch angle is not claimed as observed":
        True,

    "Empirical spin is not claimed as observed":
        True,

    "Speed-outcome relationship treated as observational":
        reference_results[
            "V9_speed_outcome_OR_per_10_kmh"
        ] > 1.0
}

# ============================================================
# 2. Display audit
# ============================================================

audit_rows = []

for name, passed in audit_checks.items():

    audit_rows.append({
        "Check": name,
        "Status": "PASS" if passed else "FAIL"
    })

audit_df = pd.DataFrame(
    audit_rows
)

display(audit_df)

# ============================================================
# 3. Overall status
# ============================================================

total_checks = len(
    audit_checks
)

passed_checks = sum(
    audit_checks.values()
)

failed_checks = (
    total_checks
    - passed_checks
)

print(
    f"\nTotal audit checks:  {total_checks}"
)

print(
    f"Passed:              {passed_checks}"
)

print(
    f"Failed:              {failed_checks}"
)

# ============================================================
# 4. Final project status
# ============================================================

all_pass = (
    failed_checks == 0
)

print("\n" + "=" * 70)

if all_pass:

    print(
        "V10 CELL 10 PASS — "
        "FINAL SCIENTIFIC AUDIT PASSED"
    )

    print(
        "\nTHE COMPUTATIONAL NOTEBOOK SERIES "
        "IS COMPLETE."
    )

else:

    print(
        "V10 CELL 10 FAIL — "
        "SCIENTIFIC AUDIT REQUIRES REVIEW"
    )

print("=" * 70)